# Intensity transformations and histograms

Explore negative images, scaling, logarithms, gamma correction, piecewise transformations, slicing, and histogram equalization.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src/image_processing").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from image_processing import load_sample, show_images

try:
    import cv2 as cv
except ImportError:
    cv = None
    print("OpenCV is unavailable. NumPy examples still run; install requirements.txt for comparisons.")

## Negative, scaling, and logarithms

Convert to floating point before arithmetic to avoid uint8 overflow.

In [ ]:
image = load_sample()
scaled = np.clip(image.astype(float) * 1.4, 0, 255).astype(np.uint8)
log_image = (255 * np.log1p(image.astype(float)) / np.log(256)).astype(np.uint8)
show_images([image, 255 - image, scaled, log_image], ["Original", "Negative", "Scaled", "Logarithm"])

## Gamma correction

Normalize to [0, 1], apply a power, then scale back to [0, 255].

In [ ]:
images = [image]
titles = ["Original"]
for gamma in [.5, 1.5, 3]:
    images.append((255 * (image / 255.0)**gamma).astype(np.uint8))
    titles.append(f"Gamma {gamma}")
show_images(images, titles)

## Piecewise contrast stretching

Map selected input intensity points to output intensity points.

In [ ]:
stretched = np.interp(image, [0, 70, 170, 255], [0, 30, 230, 255]).astype(np.uint8)
show_images([image, stretched], ["Original", "Piecewise transformation"])

## Threshold and intensity slicing

A threshold separates two intensity groups. A slice highlights a chosen interval.

In [ ]:
thresholded = np.where(image >= 120, 255, 0).astype(np.uint8)
sliced = np.where((image >= 100) & (image <= 170), 255, image).astype(np.uint8)
show_images([image, thresholded, sliced], ["Original", "Threshold", "Intensity slice"])

## Bit planes

Each bit plane isolates one binary digit of the pixel values.

In [ ]:
planes = [((image >> bit) & 1) for bit in range(7, -1, -1)]
reconstructed = sum(plane.astype(np.uint16) * (1 << bit) for plane, bit in zip(planes, range(7, -1, -1)))
np.testing.assert_array_equal(reconstructed, image)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, plane, bit in zip(axes.flat, planes, range(7, -1, -1)):
    ax.imshow(plane, cmap="gray")
    ax.set_title(f"Bit {bit}")
    ax.axis("off")
plt.show()

## Histogram and equalization

A normalized histogram sums to one. Equalization uses the cumulative distribution to remap intensities.

In [ ]:
from image_processing import equalize_histogram
histogram = np.bincount(image.ravel(), minlength=256)
probability = histogram / image.size
assert np.isclose(probability.sum(), 1)
equalized = equalize_histogram(image)
show_images([image, equalized], ["Original", "Manual equalization"])
plt.plot(probability)
plt.xlabel("Intensity")
plt.ylabel("Probability")
plt.show()
if cv is not None:
    builtin = cv.equalizeHist(image)
    print("Largest rounding difference:", np.abs(equalized.astype(int) - builtin.astype(int)).max())
    show_images([equalized, builtin], ["NumPy", "OpenCV"])